In [0]:
from pyspark.sql.functions import col

# Unity Catalog Volume paths
landing_path = "/Volumes/workspace/lab_db/lab_landing/orders/"
schema_path = "/Volumes/workspace/lab_db/lab_landing/checkpoints/lab10_orders_autoloader_schema/"
checkpoint_path = "/Volumes/workspace/lab_db/lab_landing/checkpoints/lab10_orders_autoloader_write/"

# Read incoming JSON files using Auto Loader
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
)

# Write Bronze data as Delta
(
    df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.lab_db.bronze_orders")
)